# Stage 6b: Build the analysis-ready dataset

`raw_ingestions` (weather) is the bronze layer - schema-flexible, unclean, full of overlapping snapshots. `generation_summary` is the silver layer - validated and typed, but at a different time resolution than weather. This notebook builds the gold layer: one clean, hourly, joined dataset, written to `data/processed/` (a directory the project reserved from the start but never wrote to until now).

Runs independently of `01_explore_source_data.ipynb` - re-does its own DB reads rather than depending on that notebook's in-memory state, so these can be run in any order (or just this one, on its own).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from load_db import PostgresDatabase

db = PostgresDatabase()

## Flatten every weather snapshot into one long table

Each `raw_ingestions` row's `payload` is a list of hourly readings from one ingestion run. Explode all of them into one row-per-(snapshot, hour) table, tagged with when that snapshot was taken.

In [2]:
with db.connection() as conn, conn.cursor() as cur:
    cur.execute(
        "SELECT ingested_at, payload FROM raw_ingestions WHERE source = %s ORDER BY id",
        ("weather",),
    )
    snapshots = cur.fetchall()

records = [{**reading, "ingested_at": ingested_at} for ingested_at, payload in snapshots for reading in payload]
weather_long = pd.DataFrame(records)
weather_long["index"] = pd.to_datetime(weather_long["index"], utc=True)
weather_long = weather_long.rename(columns={"index": "timestamp"})

weather_long.shape

(75, 6)

## Dedupe to one row per hour

Every scheduled run re-fetches a rolling window, so the same hour shows up in several snapshots. Keep the reading from the *most recent* snapshot for each hour - it's the freshest measurement Open-Meteo had for that hour, and later snapshots supersede earlier forecasts of the same hour rather than sitting alongside them as separate data points.

Then floor onto the hour grid with `resample("1h")` rather than trusting the raw timestamps to already land on `:00` - real Open-Meteo data does, but nothing here should *assume* that of an upstream API, and the join below needs both series on the same grid regardless.

In [3]:
weather_hourly = (
    weather_long.sort_values("ingested_at")
    .drop_duplicates(subset="timestamp", keep="last")
    .set_index("timestamp")
    .sort_index()[["temperature_2m", "wind_speed_10m", "shortwave_radiation"]]
    .resample("1h")
    .mean()
    .dropna()
)

weather_hourly.shape

(25, 3)

## Resample generation to hourly

`generation_summary` is 15-minute resolution; averaging it up to hourly means gives it the same cadence as the weather series, so the two can be joined on a shared timestamp instead of needing a nearest-match.

In [4]:
generation_df = db.query_generation_summary(country_code="IE", limit=50_000)
generation_df["timestamp"] = pd.to_datetime(generation_df["timestamp"], utc=True)

generation_hourly = (
    generation_df.set_index("timestamp")
    .sort_index()[["renewable_pct", "total_generation_mw", "renewable_mw"]]
    .resample("1h")
    .mean()
)

generation_hourly.shape

/Users/stephenquirke/Library/CloudStorage/OneDrive-Personal/Documents/developer/energy-data-pipeline/src/load_db.py:251: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(


(25, 3)

## Join and write the gold dataset

In [5]:
merged = generation_hourly.join(weather_hourly, how="inner").dropna()

output_path = Path("../data/processed/weather_generation_merged.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(output_path)

print(f"Wrote {len(merged)} rows to {output_path}")
merged.head()

Wrote 25 rows to ../data/processed/weather_generation_merged.csv


,renewable_pct,total_generation_mw,renewable_mw,temperature_2m,wind_speed_10m,shortwave_radiation
timestamp,,,,,,
2026-08-14 15:00:00+00:00,38.253333,1482.0,567.0,8.0,15.0,0.0
2026-08-14 16:00:00+00:00,39.295000,1594.0,626.5,8.5,18.0,0.0
2026-08-14 17:00:00+00:00,40.620000,1709.5,694.5,9.0,21.0,0.0
2026-08-14 18:00:00+00:00,42.357500,1800.0,762.5,9.5,24.0,40.0
2026-08-14 19:00:00+00:00,42.172500,1898.0,800.5,10.0,27.0,80.0


## A note on data volume

The scheduled workflow only just started running live, so this dataset is small early on - every 6-hour run adds another rolling window of both weather and generation data. `03_visualization.ipynb` reads whatever is in this CSV each time it's re-run, so the correlation it shows gets more meaningful as more scheduled runs land, with no code changes needed here.